# Grok-robotics-08-mbrl-mpc

**Stage 08 — Model-Based RL & MPC**

## 概念
1. **Dyna-Q**：真实交互 + 模型 imaginary rollouts 加速价值学习  
2. **MPC**：在已知/学习动力学上短视优化动作序列  
3. **样本效率**：更多数据 → 动力学预测 held-out 误差下降

## 实验
- A: 相同 episode 预算，Dyna-Q vs Q-learning（看**前半程**学习速度）
- B: Pendulum 真模型随机射击 MPC vs random
- C: 学习 f(s,a) 的 held-out MSE 随数据量变化


In [ ]:

import json, time
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

OUT=Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
np.random.seed(0); torch.manual_seed(0)
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
gpu={"cuda":torch.cuda.is_available(),"device_count":torch.cuda.device_count(),"names":[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []}
print(gpu)
t0=time.time()


In [ ]:

ACTS={0:(-1,0),1:(0,1),2:(1,0),3:(0,-1)}
class Cliff:
    def __init__(self):
        self.H,self.W=4,12; self.start=(3,0); self.goal=(3,11)
        self.cliff={(3,c) for c in range(1,11)}; self.nS=48; self.nA=4
    def sid(self,r,c): return r*self.W+c
    def reset(self):
        self.s=self.start; return self.sid(*self.s)
    def step(self,a):
        r,c=self.s; dr,dc=ACTS[a]; nr,nc=r+dr,c+dc
        if not(0<=nr<self.H and 0<=nc<self.W): nr,nc=r,c
        if (nr,nc) in self.cliff:
            self.s=self.start; return self.sid(*self.s), -100., True
        if (nr,nc)==self.goal:
            self.s=(nr,nc); return self.sid(*self.s), 10., True
        self.s=(nr,nc); return self.sid(*self.s), -1., False

def run_control(planning=0, episodes=120, seed=0):
    rng=np.random.default_rng(seed)
    env=Cliff()
    Q=np.zeros((env.nS,env.nA)); Model={}; rets=[]
    for ep in range(episodes):
        s=env.reset(); done=False; G=0; steps=0
        while not done and steps<200:
            if rng.random()<0.2:
                a=int(rng.integers(0,4))
            else:
                a=int(np.argmax(Q[s]))
            ns,r,done=env.step(a)
            Q[s,a]+=0.5*(r+(0 if done else 0.99*Q[ns].max())-Q[s,a])
            Model[(s,a)]=(ns,r,done)
            keys=list(Model.keys())
            for _ in range(planning):
                if not keys: break
                ss,aa=keys[int(rng.integers(0,len(keys)))]
                nss,rr,dd=Model[(ss,aa)]
                Q[ss,aa]+=0.5*(rr+(0 if dd else 0.99*Q[nss].max())-Q[ss,aa])
            G+=r; s=ns; steps+=1
        rets.append(G)
    return np.array(rets)

# average over seeds for stability
def multi(planning, seeds=5):
    arr=np.stack([run_control(planning, episodes=100, seed=s) for s in range(seeds)],0)
    return arr.mean(0)

ret_q=multi(0); ret_dyna=multi(50)
early_q=float(ret_q[:40].mean()); early_d=float(ret_dyna[:40].mean())
late_q=float(ret_q[-20:].mean()); late_d=float(ret_dyna[-20:].mean())
print("early Q/Dyna", early_q, early_d, "late", late_q, late_d)


In [ ]:

class Pendulum:
    def __init__(self):
        self.max_speed=8.; self.max_torque=2.; self.dt=0.05; self.g=10.; self.m=1.; self.l=1.
    def reset(self, bottom=True):
        self.state=np.array([np.pi if bottom else 0.0, 0.0], np.float32); return self.state.copy()
    def dyn(self, state, u):
        th,thdot=state; u=float(np.clip(u,-2,2))
        newthdot=thdot+(-3*self.g/(2*self.l)*np.sin(th+np.pi)+3./(self.m*self.l**2)*u)*self.dt
        newth=th+newthdot*self.dt; newthdot=np.clip(newthdot,-self.max_speed,self.max_speed)
        return np.array([newth,newthdot],np.float32)
    def step(self,u):
        th,thdot=self.state; u=float(np.clip(u,-2,2))
        cost=(((th+np.pi)%(2*np.pi))-np.pi)**2 + 0.1*thdot**2 + 0.001*u**2
        self.state=self.dyn(self.state,u)
        return self.state.copy(), -float(cost)

def mpc_true(env, H=20, N=128):
    th,thdot=env.state
    acts=np.random.uniform(-2,2,(N,H))
    best=-1e18; ba=0.0
    for i in range(N):
        st=np.array([th,thdot],np.float32); ret=0.0
        for t in range(H):
            u=acts[i,t]
            cost=(((st[0]+np.pi)%(2*np.pi))-np.pi)**2 + 0.1*st[1]**2 + 0.001*u**2
            ret += -cost
            st=env.dyn(st,u)
        if ret>best: best=ret; ba=acts[i,0]
    return float(ba)

def eval_pend(ctrl, T=200, n=5):
    sc=[]
    for _ in range(n):
        e=Pendulum(); e.reset(True); R=0
        for _ in range(T):
            a=ctrl(e); _,r=e.step(a); R+=r
        sc.append(R)
    return float(np.mean(sc))

rand_sc=eval_pend(lambda e: float(np.random.uniform(-2,2)))
mpc_sc=eval_pend(lambda e: mpc_true(e))
print("random", rand_sc, "mpc", mpc_sc)


In [ ]:

class DynNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(3,128),nn.ReLU(),nn.Linear(128,128),nn.ReLU(),nn.Linear(128,2))
    def forward(self, s, a):
        return self.net(torch.cat([s,a],-1))

def make_dataset(n):
    e=Pendulum(); X=[]; Y=[]
    e.reset(False)
    for i in range(n):
        s=e.state.copy(); a=np.random.uniform(-2,2)
        e.step(a); ns=e.state.copy()
        X.append(np.array([s[0],s[1],a],np.float32)); Y.append(ns)
        if (i+1)%100==0: e.reset(np.random.rand()>0.5)
    return np.array(X), np.array(Y)

# fixed held-out test
Xte,Yte=make_dataset(2000)
Xte_t=torch.tensor(Xte,device=device); Yte_t=torch.tensor(Yte,device=device)

def fit_mse(n_train, epochs=80):
    X,Y=make_dataset(n_train)
    m=DynNet().to(device)
    opt=torch.optim.Adam(m.parameters(), lr=1e-3)
    Xt=torch.tensor(X,device=device); Yt=torch.tensor(Y,device=device)
    for _ in range(epochs):
        pred=m(Xt[:,:2], Xt[:,2:3])
        loss=F.mse_loss(pred, Yt)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        te=float(F.mse_loss(m(Xte_t[:,:2], Xte_t[:,2:3]), Yte_t).item())
    return te

curve=[]
for n in [100,300,800,2000,5000]:
    te=fit_mse(n)
    curve.append({"n":n,"heldout_mse":te}); print(curve[-1])

elapsed=time.time()-t0


In [ ]:

fig,axes=plt.subplots(1,3,figsize=(13,3.8))
def sm(x,w=8):
    c=np.cumsum(np.insert(x.astype(float),0,0)); return (c[w:]-c[:-w])/w if len(x)>=w else x
axes[0].plot(sm(ret_q), label="Q-learning")
axes[0].plot(sm(ret_dyna), label="Dyna-Q")
axes[0].legend(); axes[0].set_title("A) Dyna accelerates learning"); axes[0].set_xlabel("episode")
axes[1].bar(["random","true MPC"],[rand_sc,mpc_sc], color=["#e76f51","#2a9d8f"])
axes[1].set_title("B) MPC with true dynamics")
axes[2].plot([c["n"] for c in curve],[c["heldout_mse"] for c in curve], "-o")
axes[2].set_title("C) Held-out dynamics MSE"); axes[2].set_xlabel("# train transitions")
fig.tight_layout(); fig.savefig(OUT/"stage08_mbrl_mpc.png", dpi=120); plt.close(fig)

payload={
  "ok": True,
  "stage": "08-mbrl-mpc",
  "title": "Grok-robotics-08-mbrl-mpc",
  "metrics": {
    "early_q": early_q, "early_dyna": early_d,
    "late_q": late_q, "late_dyna": late_d,
    "random_pendulum": rand_sc, "true_mpc_pendulum": mpc_sc,
    "model_heldout_curve": curve,
  },
  "gpu": gpu, "elapsed_sec": elapsed,
  "concept": "model-based acceleration (Dyna) + short-horizon planning (MPC)",
  "new_capability": "reuse transitions via model; plan with dynamics",
  "compare_to_previous": "Stage07 model-free continuous SAC; Stage08 adds model learning and planning",
}
# Primary pedagogical asserts
assert early_d > early_q, (early_d, early_q)  # Dyna learns faster early
assert mpc_sc > rand_sc + 50, (mpc_sc, rand_sc)
assert curve[-1]["heldout_mse"] < curve[0]["heldout_mse"], curve
(OUT/"results_stage08.json").write_text(json.dumps(payload, indent=2))
print(json.dumps(payload, indent=2))
print("STAGE08_OK")
